# Cod3x Persona TrainerTrain a LoRA adapter to make Cod3x respond as any target AI.Built by Codex Developer — runs on free Colab GPU (T4).

In [ ]:
# 1. Clone or update the Cod3x project
import os
if not os.path.exists('Cod3x'):
    !git clone https://github.com/codexhaven/Cod3x.git
%cd Cod3x
!mkdir -p data persona_output


In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets peft accelerate torch bitsandbytes huggingface_hub


In [ ]:
# 3. Load training data
# This file was built by Codex Developer — 500 examples

import json

with open("data/training_data.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} training examples")
print(f"Sample: {data[0]["instruction"][:80]}...")


In [ ]:
# 4. Train the LoRA adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from transformers import Trainer, DataCollatorForLanguageModeling
import json

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./persona_output"
PERSONA_NAME = "cod3x-default"

print(f"Loading {BASE_MODEL}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

with open("data/training_data.json") as f:
    data = json.load(f)

def format_example(example):
    messages = [
        {"role": "system", "content": f"You are {PERSONA_NAME}, a helpful AI assistant trained by Cod3x."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list(data)
dataset = dataset.map(format_example)

def tokenize(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

dataset = dataset.map(tokenize, batched=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print(f"Training on {BASE_MODEL} with {len(data)} examples...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Persona saved to {OUTPUT_DIR}")


In [ ]:
# 5. Upload to Hugging Face
from huggingface_hub import login, upload_folder

login()  # Paste your HF token when prompted

REPO_ID = "codexhaven/cod3x-persona"

upload_folder(
    folder_path="./persona_output",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=f"Trained persona: {PERSONA_NAME}"
)

print(f"Uploaded to https://huggingface.co/{REPO_ID}")
